# 01: Unsupervised Clustering (K-Means, DBSCAN & Hierarchical Agglomerative)

**Track 06: Unsupervised Learning, Clustering & Dimension Reduction** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Customer segmentation using K-Means++ initialization, optimal $k$ search (Inertia Elbow & Silhouette Score), density-based DBSCAN, and Agglomerative Hierarchical dendrograms.


## 1. Load Customer Segmentation Dataset & Standardize
Ingest customer behavioral metrics: Annual Income, Spending Score, Recency, Frequency.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score

df = load_dataset("customer_segmentation")
num_cols = df.select_dtypes(include=[np.number]).columns

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[num_cols])

print(f"Customer records: {df.shape[0]}, Features: {len(num_cols)}")

## 2. Finding Optimal Clusters ($k$) with Silhouette Analysis
Evaluate $k \in [2, 7]$ to balance cluster cohesion and separation.

In [ ]:
silhouette_scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append((k, score))

optimal_k, best_score = max(silhouette_scores, key=lambda item: item[1])
print("=== Silhouette Analysis Across K ===")
for k, s in silhouette_scores:
    print(f"K = {k} -> Silhouette Score: {s:.4f}")
print(f"\nOptimal K Selected: {optimal_k} with score {best_score:.4f}")

## 3. Density-Based Clustering with DBSCAN
Identify natural non-spherical clusters and noise/outliers without prespecifying $k$.

In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_db = list(db_labels).count(-1)

print("=== DBSCAN Density Clustering Results ===")
print(f"Estimated Clusters Formed: {n_clusters_db}")
print(f"Detected Noise Points    : {n_noise_db} ({n_noise_db/len(X_scaled)*100:.2f}%)")